In [2]:
import numpy as np
import pandas as pd
from meteostat import daily
import pandas as pd
from datetime import date


In [ ]:
weights_path = "/mnt/data/dep_station_KNN_weights.csv"

w = pd.read_csv(
    weights_path,
    dtype={"departement_code": str, "station_id": str}
)

# Basic sanity checks
required_cols = {"departement_code", "station_id", "weight"}
missing = required_cols - set(w.columns)
if missing:
    raise ValueError(f"weights CSV missing columns: {missing}")

# Optional: ensure weights sum ~ 1 by department
sums = w.groupby("departement_code")["weight"].sum()
max_err = (sums - 1.0).abs().max()
print("Max |sum(weights)-1| across departments =", max_err)

# dict: dep -> pd.Series(index=station_id, values=weight)
dep_weights = {
    dep: g.set_index("station_id")["weight"].astype(float)
    for dep, g in w.groupby("departement_code", sort=True)
}


In [ ]:


# 1) List of unique stations used by your KNN weights
station_ids = sorted(w["station_id"].unique())

# 2) Choose your historical range
start = date(2000, 1, 1)
end   = date(2025, 12, 31)

# 3) Pull daily data per station and combine
dfs = []
for sid in station_ids:
    df = daily(sid, start, end).fetch()  # index = time (datetime), columns include tavg, tmin, tmax, prcp, etc.
    if df is None or df.empty:
        continue
    df = df.reset_index()  # move time index into a column named 'time'
    df["station_id"] = sid
    dfs.append(df[["time", "station_id", "tavg"]])

station_long = pd.concat(dfs, ignore_index=True)

# 4) Build station_daily_tavg: index=date, columns=station_id, values=tavg
station_daily_tavg = (
    station_long
    .rename(columns={"time": "date"})
    .pivot(index="date", columns="station_id", values="tavg")
    .sort_index()
)

# Optional: ensure daily frequency on index (you can reindex if you want full grid)
station_daily_tavg.index = pd.to_datetime(station_daily_tavg.index)
station_daily_tavg.columns.name = "station_id"

print(station_daily_tavg.shape)
station_daily_tavg.head()


In [3]:
# ---------- 1) Department aggregation (your KNN weights) ----------
def dept_daily_temp(station_daily_tavg: pd.DataFrame, dep_weight_series: pd.Series) -> pd.Series:
    common = [sid for sid in dep_weight_series.index if sid in station_daily_tavg.columns]
    X = station_daily_tavg[common]
    W = dep_weight_series.loc[common]

    avail = ~X.isna()
    W_avail = avail.mul(W, axis=1)
    denom = W_avail.sum(axis=1).replace(0, np.nan)

    T_dep = (X.fillna(0) * W_avail).sum(axis=1) / denom
    return T_dep

# ---------- 2) Seasonality fit ----------
def add_time_features(df, date_col="date"):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col)
    df["doy"] = df[date_col].dt.dayofyear.astype(int)
    df["sin1"] = np.sin(2 * np.pi * df["doy"] / 365.25)
    df["cos1"] = np.cos(2 * np.pi * df["doy"] / 365.25)
    df["t"] = (df[date_col] - df[date_col].min()).dt.days.astype(float)
    return df

def fit_seasonal_mean(df, temp_col="temp", include_trend=True):
    X_cols = ["sin1", "cos1"]
    if include_trend:
        X_cols = ["t"] + X_cols

    X = np.column_stack([np.ones(len(df))] + [df[c].values for c in X_cols])
    y = df[temp_col].values
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    S = X @ beta

    def seasonal_mean_func(new_dates: pd.DatetimeIndex):
        new_df = pd.DataFrame({"date": new_dates})
        new_df = add_time_features(new_df, "date")
        Xn = np.column_stack([np.ones(len(new_df))] + [new_df[c].values for c in X_cols])
        return Xn @ beta

    return beta, S, seasonal_mean_func

# ---------- 3) OU (AR1) fit on residuals ----------
def fit_ou_ar1(residuals):
    x = residuals[:-1]
    y = residuals[1:]
    phi = np.dot(x, y) / np.dot(x, x)
    phi = np.clip(phi, 1e-6, 0.999999)

    eps = y - phi * x
    sigma_eps = eps.std(ddof=1)
    kappa = -np.log(phi)  # Δ=1 day

    return phi, sigma_eps, kappa

# ---------- 4) Simulation ----------
def simulate_temperatures(dates, seasonal_mean_func, phi, sigma_eps, x0, n_paths=20000, seed=0):
    rng = np.random.default_rng(seed)
    dates = pd.to_datetime(dates)
    n = len(dates)

    S = seasonal_mean_func(dates)

    X = np.empty((n_paths, n), dtype=float)
    T = np.empty((n_paths, n), dtype=float)

    X[:, 0] = x0
    T[:, 0] = S[0] + X[:, 0]

    shocks = rng.normal(0.0, sigma_eps, size=(n_paths, n - 1))
    for t in range(1, n):
        X[:, t] = phi * X[:, t - 1] + shocks[:, t - 1]
        T[:, t] = S[t] + X[:, t]

    return T  # shape: (paths, days)

# ---------- 5) Indices ----------
def CAT_index(T_paths):
    return T_paths.sum(axis=1)

def HDD_index(T_paths, base=10.0):
    return np.maximum(base - T_paths, 0.0).sum(axis=1)

# ---------- 6) Pricing ----------
def price_future(index_samples, r=0.0, tau_years=0.0):
    return np.exp(-r * tau_years) * index_samples.mean()

def price_call(index_samples, strike, notional=1.0, r=0.0, tau_years=0.0):
    payoff = np.maximum(index_samples - strike, 0.0) * notional
    return np.exp(-r * tau_years) * payoff.mean()

def price_put(index_samples, strike, notional=1.0, r=0.0, tau_years=0.0):
    payoff = np.maximum(strike - index_samples, 0.0) * notional
    return np.exp(-r * tau_years) * payoff.mean()

# ---------- 7) One department: fit + simulate + price ----------
def ou_price_department(
    station_daily_tavg, dep_weight_series,
    window_start, window_end,
    hdd_base=10.0,
    include_trend=True,
    n_paths=50000,
    seed=0,
    r=0.0
):
    # Build dept daily temp
    T_dep = dept_daily_temp(station_daily_tavg, dep_weight_series).dropna()
    df = pd.DataFrame({"date": T_dep.index, "temp": T_dep.values})
    df = add_time_features(df, "date")

    # Fit seasonal mean + OU
    _, S, seasonal_mean_func = fit_seasonal_mean(df, "temp", include_trend=include_trend)
    resid = df["temp"].values - S
    phi, sigma_eps, kappa = fit_ou_ar1(resid)

    # Simulate for the contract window
    dates = pd.date_range(pd.to_datetime(window_start), pd.to_datetime(window_end), freq="D")
    x0 = resid[-1]  # last observed residual
    T_paths = simulate_temperatures(dates, seasonal_mean_func, phi, sigma_eps, x0, n_paths=n_paths, seed=seed)

    # Build indices
    cat = CAT_index(T_paths)
    hdd = HDD_index(T_paths, base=hdd_base)

    tau_years = len(dates) / 365.25
    cat_fut = price_future(cat, r=r, tau_years=tau_years)
    hdd_fut = price_future(hdd, r=r, tau_years=tau_years)

    return {
        "phi": phi, "kappa": kappa, "sigma_eps": sigma_eps,
        "CAT_mean": float(cat.mean()), "CAT_std": float(cat.std(ddof=1)), "CAT_future": float(cat_fut),
        "HDD_mean": float(hdd.mean()), "HDD_std": float(hdd.std(ddof=1)), "HDD_future": float(hdd_fut),
    }
